In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set up clean plotting styles for Jupyter
%matplotlib inline
sns.set_theme(style="whitegrid")

# Load the file (replace with your exact local path if necessary)
df = pd.read_parquet("0000-2.parquet") #### GDPX2 dataset

# Display the first few rows to confirm loading
df.head()

#### Select active compounds to look up

compounds = df[~df['sample_type'].str.contains('Control', na=False)]['compound'].unique().tolist()
open('./BioXAI_Hackathon/active_compounds_1263.txt', 'w').write('\n'.join(df[~df['sample_type'].str.contains('Control', na=False)]['compound'].dropna().unique().astype(str)))

# compounds('./BioXAI_Hackathon/active_compounds.csv', index = False)

print(f"Found {len(compounds)} unique active compounds to look up.")

Found 1263 unique active compounds to look up.


In [3]:
len(compounds)

1263

In [ ]:
import requests
import urllib.parse
import pandas as pd
import time

def query_pubchem(name):
    """Queries PubChem PUG REST API for Canonical SMILES."""
    # URL-encode the name to safely handle symbols like (±), hyphens, and spaces
    encoded_name = urllib.parse.quote(name)
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{encoded_name}/property/CanonicalSMILES/JSON"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            return data['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except Exception:
        pass
    return None

def query_chembl_robust(name):
    """Fallback query to ChEMBL API targeting primary names and synonyms."""
    base_url = "https://www.ebi.ac.uk/chembl/api/data/molecule.json"
    
    # Try preferred name first
    try:
        res = requests.get(base_url, params={'pref_name__iexact': name, 'format': 'json'}, timeout=5)
        if res.status_code == 200 and res.json().get('molecules'):
            structs = res.json()['molecules'][0].get('molecule_structures', {})
            if structs and structs.get('canonical_smiles'):
                return structs['canonical_smiles']
    except Exception:
        pass
        
    # Try nested synonyms array
    try:
        res = requests.get(base_url, params={'molecule_synonyms__molecule_synonym__iexact': name, 'format': 'json'}, timeout=5)
        if res.status_code == 200 and res.json().get('molecules'):
            structs = res.json()['molecules'][0].get('molecule_structures', {})
            if structs and structs.get('canonical_smiles'):
                return structs['canonical_smiles']
    except Exception:
        pass
    return None

# 2. Execute the mapping loop
smiles_results = {}

# print("Starting molecular structure resolution...")
for compound in compounds:
    # Attempt primary lookup with PubChem (handles stereochemistry prefixes beautifully)
    smiles = query_pubchem(compound)
    
    # If PubChem fails (e.g. due to specific syntax), try cleaning the prefix as a fallback
    if not smiles and compound.startswith('(±)-'):
        cleaned_name = compound.replace('(±)-', '')
        smiles = query_pubchem(cleaned_name)
        
    # If still not found, cross-reference with ChEMBL
    if not smiles:
        smiles = query_chembl_robust(compound)
        
    smiles_results[compound] = smiles
    # print(f"Mapped: {compound} -> {smiles if smiles else 'NOT FOUND'}")
    
    # Polite pause between requests
    time.sleep(0.2)

# 3. Format the results cleanly into a Pandas DataFrame
results_df = pd.DataFrame(list(smiles_results.items()), columns=['Compound_Name', 'SMILES'])
results_df.to_csv('./BioXAI_Hackathon/smiles_df_for_compounds.csv', index = False)
results_df

Starting molecular structure resolution...
Mapped: (±)-Vanillylmandelic acid -> NOT FOUND
Mapped: L-2-aminoadipic acid -> NOT FOUND
Mapped: 2-methoxyestradiol -> COc1cc2c(cc1O)CC[C@@H]1[C@@H]2CC[C@]2(C)[C@@H](O)CC[C@@H]12
Mapped: Oleic Acid -> CCCCCCCC/C=C\CCCCCCCC(=O)O
Mapped: 4-Hydroxy-3-methoxyphenylacetic acid -> NOT FOUND
Mapped: L-Aspartic acid -> N[C@@H](CC(=O)O)C(=O)O
Mapped: D-Serine -> N[C@H](CO)C(=O)O
Mapped: 5-Fluorouracil -> O=c1[nH]cc(F)c(=O)[nH]1
Mapped: Melatonin -> COc1ccc2[nH]cc(CCNC(C)=O)c2c1
Mapped: 4-Imidazoleacrylic acid -> NOT FOUND
Mapped: N-Methyl-D-aspartic acid -> NOT FOUND
Mapped: 1,7-Dimethylxanthine -> NOT FOUND
Mapped: Theobromine -> Cn1cnc2c1c(=O)[nH]c(=O)n2C
Mapped: N-Acetyl-L-Cysteine -> CC(=O)N[C@@H](CS)C(=O)O
Mapped: Taurine -> NCCS(=O)(=O)O
Mapped: Theophylline -> Cn1c(=O)c2nc[nH]c2n(C)c1=O.O
Mapped: Uridine 5'-diphosphate sodium -> NOT FOUND
Mapped: Phosphonoacetic acid -> NOT FOUND
Mapped: L-Arginine -> N=C(N)NCCC[C@H](N)C(=O)O
Mapped: Forskolin -